# Lesson 2.5: Trace call identity, scope and incremental output

Read the matching lesson at `modules/02-conversation/05-source-walkthrough.md`. Run this notebook from top to bottom in the project `.venv` kernel. All cases are fictional. The default path sends no model-provider requests. A kernel retains values between cells; restart and run all after an experiment to check independence.

## The practical question

A conversation can appear coherent while the application has lost the selected case or confused two users. Harbour therefore keeps the selected case as explicit state and treats each tool request as a proposal that still needs validation.

In [ ]:
from ai_orgdesign.labs import m02 as lab
print("Module 2 code loaded; no provider called.")

## Follow the mechanism

The conversation function creates a fresh history list for each call. A scripted call carries an identifier, a capability name and an arguments dictionary. dispatch permits only lookup_case with exactly one case_id field, then lookup checks the tenant before returning a copy of the record. Serialising that result makes the tool observation inspectable without changing the authoritative record. stream_words yields one word at a time; it demonstrates incremental delivery without pretending that a model produced the words.

In [ ]:
import inspect
print(inspect.getsource(lab.conversation))

In [ ]:
import inspect
print(inspect.getsource(lab.stream_words))

## Predict and run

The selected case is H-17, its ordered quantity is twelve and its reported quantity is ten. The tool observation has call_id equal to call-1. Joining the stream pieces reconstructs Evidence awaits review, with a trailing space. Nothing approves a remedy.

In [ ]:
from ai_orgdesign.labs.m02 import conversation, stream_words
result = conversation()
assert result["state"]["selected_case"] == "H-17"
assert "".join(stream_words("Evidence awaits review")).strip() == "Evidence awaits review"
print(result)

## Exercise

Request H-18 while keeping the north scope. Predict whether the record will be returned and whether changing a name inside a model message should alter that decision.

Pause and write the predicted result and its interpretation before opening the worked answer below.

## Worked answer

lookup raises PermissionError because H-18 belongs to south. Identity and scope enter through the application, not the request text. A real authentication system would supply that trusted scope; this local function parameter is a teaching boundary, not authentication.

In [ ]:
from ai_orgdesign.business import dispatch
try:
    dispatch("lookup_case", {"case_id": "H-18"}, "north")
except PermissionError:
    print("Expected: cross-scope read denied")
else:
    raise AssertionError("Scope boundary failed")

## Alternatives and organisational meaning

A form with a required case identifier may be more dependable than extracting it from chat. A provider adapter can propose the same tool call, but should not bypass dispatch. Streaming can improve perceived responsiveness while adding partial-output and cancellation problems.

Explain separately what executed, what its output establishes and what remains a question for the surrounding service.

## Optional live provider call

This cell is disabled by default. Enabling it sends fictional case context to the configured provider and may incur charges. Set OPENAI_API_KEY and OPENAI_MODEL in the root .env first. The model must support the function-calling interface used by the adapter. The offline protocol test does not prove live account access or output quality.

In [ ]:
RUN_PAID_PROVIDER = False
if RUN_PAID_PROVIDER:
    from ai_orgdesign.live import ask
    print(ask("Inspect fictional case H-17 and preserve unresolved facts.", consent=True))
else:
    print("Provider call skipped; no request sent.")